In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken
import truststore
import urllib.request
import os

truststore.inject_into_ssl()

# Tokenizer (Byte Pair Encoding)


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
)
tokens = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(tokens)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [5]:
tokenizer.encode("someunknownPlace", allowed_special={"<|endoftext|>"})

[11246, 34680, 27271]

Here, the BPE tokenizer represents an unknown word as a array of part words. BPE can handle unknown words by breaking them into existing known tokens.


In [6]:
tokens = tokenizer.decode(tokens)
print(tokens)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


## Masking/sliding window


We need to create input-target pairs by splitting a sentence into input LLM receives, target to predict (one word), and the rest of the sentence the LLM can't see.


In [ ]:
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "the-verdict.txt"
if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)


In [8]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [9]:
enc_text = tokenizer.encode(raw_text)
print("Total number of tokens:", len(enc_text))

Total number of tokens: 5145


Below is how the input-target pair looks like:


In [ ]:
enc_sample = enc_text[50:]
context_size = 4
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


## Data Loader with Pytorch

We need to create input tensor and target tensor of the same size. They are offset by 1. They are packaged in dataset.


In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader


In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)


[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


## creating token embeddings

Tokens IDs need to be transformed into embeddings vectors by multiplying the token id with a **embedding layer**, which are initialized as random values. Embedding layers function as loop up tables.


In [13]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print("embedding layer:")
print(embedding_layer.weight)
print("embedding vector of token 3:")
print(embedding_layer(torch.tensor([3])))
print("embedding vectors of the input tokens:")
print(embedding_layer(input_ids))

embedding layer:
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
embedding vector of token 3:
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
embedding vectors of the input tokens:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## positions

Embedding vector lookup process above doesn't involve positions. Self-attention is also position agnostic. We want to add some positional information to embeddings. There two types of position embeddings: absolute position embeddings and relative position embeddings. Here we use absolute position embedding, where each token (at different position) will be summed with a fixed vector to signify its position.


In [ ]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)
token_embeddings = token_embedding_layer(inputs)
print("tokens are converted to token embeddings with shape:")
print(token_embeddings.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])
tokens are converted to token embeddings with shape:
torch.Size([8, 4, 256])


The shape of the token embedding is 8 x 4 x 256. 8 is the batch number. 4 is context length. 256 is the size of the embedding vector. The size of the position embedding should be the same. For each input in the batch, we add a 4 x 256 shaped position embeddings. The `pos_embedding_layer` is initialized with a vector from 0 to length - 1. `context_length` doesn't have to be the `max_length` supported by the model. If you want to process a longer text, just need to truncate them.


In [15]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings[:5][:])
print(pos_embeddings.shape)
print("final embedding after adding positional information:")
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

tensor([[ 1.7375, -0.5620, -0.6303,  ..., -0.2277,  1.5748,  1.0345],
        [ 1.6423, -0.7201,  0.2062,  ...,  0.4118,  0.1498, -0.4628],
        [-0.4651, -0.7757,  0.5806,  ...,  1.4335, -0.4963,  0.8579],
        [-0.6754, -0.4628,  1.4323,  ...,  0.8139, -0.7088,  0.4827]],
       grad_fn=<SliceBackward0>)
torch.Size([4, 256])
final embedding after adding positional information:
torch.Size([8, 4, 256])


# Attention Mechanism

We will cover 4 variants of attention mechanism: simplified self-attention, self-attention, causal attention and multi-head attention.

In a translation task, one often needs to look ahead in the source sentence to understand how to translate the sentence as a whole before translating the first word, this often involves a encoder-decoder architecture. Before LLM, the most popular encoder-decoder architecture is RNN. RNN uses hidden states to memorize the entire sentence during the encoding phase, then it produces the translation one word at a time, updating the hidden states at every word. A problem of RNN is that they don't have access to previous words in the input.

Self-attention builds on a variant of RNN but with a new architecture. It allows each position in the input to consider all other positions.


The goal of self-attention is to compute a context vector for each input element that combines information from all other input elements. In an input sequence from $x_1$ to $x_t$, each is represented by a $d$-dimensional vector. Self-attention calculate context vector $z_i$ for each element $x_i$. A **context vector** is an enriched embedding vector that incorporate information from all other elements in the sequence.


### Simplified Self-Attention

In this example, we want to calculate context vector $z_2$ for the word "journey" in the sentence "Your journey starts with one step". To do so, we will need to calculate intermediate attention score $\omega_i$ where $\omega_i = x_2 \cdot x_i$. Notice that the attention score the dot product of the target tensor with itself. Since dot product is a measure of similarity between vectors, we can understand attention as a similarity measure.


In [ ]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your (x^1)
        [0.55, 0.87, 0.66],  # journey (x^2)
        [0.57, 0.85, 0.64],  # starts (x^3)
        [0.22, 0.58, 0.33],  # with (x^4)
        [0.77, 0.25, 0.10],  # one (x^5)
        [0.05, 0.80, 0.55],
    ]  # step (x^6)
)
print('"journey" is: ', inputs[1])

"journey" is:  tensor([0.5500, 0.8700, 0.6600])


In [ ]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print('attention score of "journey" with all other words: ', attn_scores_2)

attention score of "journey" with all other words:  tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [18]:
torch.dot(inputs[1], inputs[1])

tensor(1.4950)

Next, we need to normalize the attention scores to get the scores that sum to 1. Now they are called **attention weights**


In [19]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


Alternatively, we can use the `softmax` function as a normalization method. It manages extreme values better. It ensures all elements are positive, which make them more interpretable. Here, higher weights mean greater importance.


In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)


attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [21]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights using torch.softmax:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights using torch.softmax: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


Finally, to calculate the context vector $z_2$, we sum the product of embedding tokens $x_i$ and its attention weights: $z_2 = \sum_i a_{2i} \, x_i$:


In [ ]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [23]:
attn_weights_2[1]

tensor(0.2379)

Next, we will compute attention for all input tokens. The process of computing the self-attention is the following, we generalize it to all tokens in a sequence:

1. compute attention score ($\omega_{ki} = x_k \cdot x_i$)
2. compute attention weights ($a$), a normalized version of the attention score
3. compute context vectors ($z_k = \sum_i a_{ki} \, x_i$)

Below is the naive implementation: first we calculate the attention score


In [24]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


The above implementation is in fact matrix multiplication, which can be representation by `@` in pytorch.


In [25]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Next, we normalize the scores to get attention weights:


In [26]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


Finally, we compute the context vectors, notice that the final context vector for row 2 (token "journey") is the same as the one we calculated above.


In [27]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


Observation: suppose a sequence has $n$ elements and token embeddings has dimension $m$. The attention score/weights have shape $n \times n$. Each input token gets a dot product with all input tokens in the sequence, so we have $n$ products, together they constitute attention weights of one token. Since we have $n$ input tokens, we get an $n x n$ matrix. Alternatively, since attention score is the product of input and its transpose, we have $(n \times m)(m \times n)= n \times n$

The context vector has shape $n \times m$. It has the same shape as the input sequence. Since context vector are attention weights times input sequence, and the weight matrix doesn't change the shape of the input sequence.


### Self-attention with trainable weights

The most important difference is the introduction of weight matrices that are updated during model training. To do so, we introduce 3 trainable objects $W_q$, $W_k$, $W_v$. They project the embedded token, $x_i$, into query, key and value vectors. These weight matrices (**weight parameter**) are trainable weights, not attention weights. In the example below, we show how to compute context vector $z_2$. A caveat, the `d_out` is 2 rather than 3, like above. In GPT models, `d_in` and `d_out` are the same, but to illustrate the algorithm, we use a different value.

This demonstrate the classic **scaled dot-product attention** or self-attention introduced in the Attention is All You Need paper. The attention function is the following:

$$
    \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$


In [ ]:
x_2 = inputs[1]  # input element
d_in = inputs.shape[1]  # input embedding dimension
d_out = 2  # output embedding

# weight matrices
torch.manual_seed(123)
W_query = torch.nn.Parameter(
    torch.rand(d_in, d_out), requires_grad=False
)  # query object
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)  # key object
W_value = torch.nn.Parameter(
    torch.rand(d_in, d_out), requires_grad=False
)  # value object

# corresponding query, key, value vectors
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)
print(key_2)
print(value_2)


tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


To get $v_2$, we need all parameter matrices from all input elements:


In [29]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


Now we are in a position to calculate the unnormalized attention score for query "journey" with regard to itself, $\omega_{22}$


In [30]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


We generalize this to all input via matrix multiplication:


In [31]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


We apply normalization, here we scale the scores by the square root of keys. We scale the scores to avoid small gradients after apply softmax.


In [32]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Finally, we calculate the context vector by multiply attention weights with the value matrix:


In [33]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


Q: Why do we have 3 matrix, k, v, q, whereas in the previous section, we only have the input matrix? Is this some sort of decomposition?
Claude: This is a great conceptual question. It's not a decomposition in the traditional mathematical sense — it's better understood as a separation of roles.

**What simplified self-attention does**

In the simplified version, the input matrix X plays triple duty:

- $XX^T$ for attention scores — X acts as both query and key
- $\text{softmax}(XX^T) \cdot X$ for context vectors — X acts as values too

The same representation is used for everything. There's no freedom; similarity between tokens and what gets aggregated are both fully determined by the raw embeddings.

**What Q, K, V add**

The three weight matrices are independent linear projections of X:

$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$

They separate three conceptually distinct roles:

- **Q (query)**: "what is this token looking for?"
- **K (key)**: "what does this token advertise/offer to other tokens' queries?"
- **V (value)**: "what information does this token actually contribute when selected?"

The attention score $QK^T$ decides _how much_ to attend, and V decides _what_ to aggregate. In simplified attention, both are entangled in the same raw X.

**Why not a decomposition?**

A mathematical decomposition (like SVD or eigendecomposition) expresses the _same_ object in a different form. Here, W_Q, W_K, W_V are independent trainable parameters — they don't reconstruct X, they transform X into three different learned subspaces. So it's more accurately an **overparameterization**: giving the model the flexibility to independently learn the matching function (Q·K^T) and the content function (V).

**The intuition from information retrieval**

The Q/K/V naming actually comes from database/search concepts: a query searches against keys to find relevance scores, then retrieves values. Simplified attention collapses keys and values into the same thing, which is a strict limitation.


Next, we organize this into a class. Notice here we implement the actual attention function in the paper in a general form.


In [ ]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T  # omega
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec


In [35]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Instead of initialize the weights manually, we can use `nn.Linear`:


In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [37]:
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)


### hiding future words with causal attention

**masked attention**, or causal attention, is a special form of self-attention where model only have access to the previous and current tokens in a sequence as oppose to the entire sequence. In a matrix, the tasks forms a diagonal from top left to bottom right. Tokens above the diagonal are masked.

First we implement a basic version of the masked attention weights.


In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1362, 0.1730, 0.1736, 0.1713, 0.1792, 0.1666],
        [0.1359, 0.1730, 0.1735, 0.1716, 0.1790, 0.1670],
        [0.1366, 0.1729, 0.1734, 0.1714, 0.1788, 0.1669],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.1732, 0.1674],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.1649],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


Define the mask:


In [39]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


Apply mask to attention weights:


In [ ]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1362, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1730, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1366, 0.1729, 0.1734, 0.0000, 0.0000, 0.0000],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.0000, 0.0000],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<MulBackward0>)


Renormalize the maksed matrix to sum each row to 1


In [41]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)


We can make it more efficient by leveraging the property of the `softmax` function. Since `softmax` turns its input into a probability distribution, it will turn negative infinity as 0s. So we mask the attention scores with negative infinity after normalization, then the whole matrix will be turn into a masked attention automatically.


In [42]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[-0.2327,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2396,  0.1015,    -inf,    -inf,    -inf,    -inf],
        [-0.2323,  0.1004,  0.1045,    -inf,    -inf,    -inf],
        [-0.1344,  0.0502,  0.0523,  0.0470,    -inf,    -inf],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,    -inf],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MaskedFillBackward0>)


In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


Next, we introduce the notion of **dropout**, a technique where randomly selected units are ignored during training. This prevents overfitting. It is applied during training. In transformer architectures, we use dropout after applying the attention weights to the value vectors.


In [44]:
# create an dropout mask with a dropout rate of 50%
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


If we apply dropout with a rate of 50%, we need to increase the value of the remaining attention weights by $1/0.5 = 2$ . Which is why the tensor of 1s above becomes 2 after applying dropout. We will do the same for our attention weight.


In [45]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5659, 0.7160, 0.7181, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5159, 0.5167, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4047, 0.0000, 0.3993, 0.0000, 0.0000],
        [0.0000, 0.3430, 0.3437, 0.3434, 0.3516, 0.0000]],
       grad_fn=<MulBackward0>)


Finally, we put them this in a causal attention class that handles batch input.


In [46]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # use this to ensure buffer is moved to the same device as the model
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # we apply transpose to ensure the multiplication has the shape (batch, num_tokens, d_out) @ (b, d_out, num_tokens)
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

Below is an example. Notice that the output has the shape (batch_number, num_tokens, d_out), which is expected.


In [48]:
torch.manual_seed(123)
context_length = batch.shape[1]
print(d_in, d_out, context_length)
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

3 2 6
context_vecs.shape: torch.Size([2, 6, 2])


### Multi-headed attention

In practical terms, implementing multi-head attention involves creating multiple instances of the self-attention mechanism, each with its own weights, and then combining their outputs.


In [66]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

Notice here we use `head(x)`.This calls the `forward()` method of each head. In PyTorch (and Python in general), when you call a module instance like a function, Python's `__call__` method is invoked, which in turn calls the `forward()` method.

Using this multi-head attention wrapper, the output of each head are concatenated, so the final dimension is `d_out x num_heads`.


In [67]:
torch.manual_seed(123)
context_length = batch.shape[1]  # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


In the above implementation, each head is calculated sequentially, we can process them in parallel using matrix multiplication. We do so by using a single class, rather than a causal attention and a multi-head wrapper. In this new multi-head class, we split the input into multiple heads and reshape the key, query and value tensors.


In [68]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

Note that most of the splitting is done in the following lines:

```{python}
keys = keys.transpose(1, 2)
queries = queries.transpose(1, 2)
values = values.transpose(1, 2)

attn_scores = queries @ keys.transpose(2, 3)

...
context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
```

First, we transpose the k, q, v vector to move `num_head` from position 2 to 1, this then allow us to perform matrix multiplication between queries and keys without affecting `num_heads`. The attention scores are calculated as if they each attention head gets a separate matrix. Here, the `@` operator performs num_heads separate matrix multiplications in parallel, one for each head, treating (b, num_heads) as batch dimensions, this is Pytorch's **broadcasting** at work.

Finally we combine them in the context vector.


In [69]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
